In [ ]:
from torch import float32, no_grad, randn
from torch.nn import Linear, Module
from torch.optim import SGD

import import_ipynb
from common import T # type: ignore
from backward import BinaryCrossEntropy, Linear, Sigmoid # type: ignore

In [ ]:
class PerceptronLog(Module):
    def __init__(self, in_features):
        super().__init__()

        #
        # For demonstration purposes
        #

        self.linear = Linear(in_features, 1)
        self.sigmoid = Sigmoid()

        #
        # [Linear Layer] + [BinaryCrossEntropyWithLogits = Sigmoid + BinaryCrossEntropy] 
        # is more numerically stable than [Linear] + [Sigmoid] + [BinaryCrossEntropy]
        #

    def forward(self, x):
        self.z = self.linear(x)
        self.p = self.sigmoid(self.z)
        return self.p

In [ ]:
def _test_perceptron_log(X, Y, epochs):
    model = PerceptronLog(X.shape[1])
    optimizer = SGD(model.parameters(), lr=0.1)

    for _ in range(epochs):
        optimizer.zero_grad()

        P = model(X)
        loss = BinaryCrossEntropy()(P, Y)
        
        loss.backward()
        optimizer.step()

    return ((model(X) > 0.5) == Y).mean(dtype=float32).item()


def _test_perceptron_log_boolean(function, samples, epochs):
    X = (randn(samples, 2, dtype=float32) > 0).float()
    Y = function(X).unsqueeze(1)

    return _test_perceptron_log(X, Y, epochs)


if __name__ == "__main__":
    f_and = lambda X: X[:, 0] * X[:, 1]
    print(f"AND, 100 epochs: {_test_perceptron_log_boolean(f_and, 100, 100):.2f}") # AND, 100 epochs: 0.70
    print(f"AND, 200 epochs: {_test_perceptron_log_boolean(f_and, 100, 200):.2f}") # AND, 200 epochs: 1.00
    print(f"AND, 500 epochs: {_test_perceptron_log_boolean(f_and, 100, 500):.2f}") # AND, 500 epochs: 1.00

    print()

    f_or = lambda X: X[:, 0] + X[:, 1] - X[:, 0] * X[:, 1]
    print(f"OR, 100 epochs: {_test_perceptron_log_boolean(f_or, 100, 100):.2f}") # OR, 100 epochs: 0.71
    print(f"OR, 200 epochs: {_test_perceptron_log_boolean(f_or, 100, 200):.2f}") # OR, 200 epochs: 0.78
    print(f"OR, 500 epochs: {_test_perceptron_log_boolean(f_or, 100, 500):.2f}") # OR, 500 epochs: 1.00

    print()

    f_nand = lambda X: 1 - (X[:, 0] * X[:, 1])
    print(f"NAND, 100 epochs: {_test_perceptron_log_boolean(f_nand, 100, 100):.2f}") # NAND, 100 epochs: 0.79
    print(f"NAND, 200 epochs: {_test_perceptron_log_boolean(f_nand, 100, 200):.2f}") # NAND, 200 epochs: 1.00
    print(f"NAND, 500 epochs: {_test_perceptron_log_boolean(f_nand, 100, 500):.2f}") # NAND, 500 epochs: 1.00

    print()

    # XOR is not linearly separable, so the logistic perceptron should not be able to learn it

    f_xor = lambda X: (X[:, 0] + X[:, 1]) % 2
    print(f"XOR, 100 epochs: {_test_perceptron_log_boolean(f_xor, 100, 100):.2f}") # XOR, 100 epochs: 0.56
    print(f"XOR, 200 epochs: {_test_perceptron_log_boolean(f_xor, 100, 200):.2f}") # XOR, 200 epochs: 0.83
    print(f"XOR, 500 epochs: {_test_perceptron_log_boolean(f_xor, 100, 500):.2f}") # XOR, 500 epochs: 0.54

AND, 100 epochs: 0.78
AND, 200 epochs: 0.80
AND, 500 epochs: 1.00

OR, 100 epochs: 0.75
OR, 200 epochs: 0.77
OR, 500 epochs: 1.00

NAND, 100 epochs: 0.79
NAND, 200 epochs: 0.80
NAND, 500 epochs: 1.00

XOR, 100 epochs: 0.55
XOR, 200 epochs: 0.54
XOR, 500 epochs: 0.58
